In [ ]:
from pyspark.sql import SparkSession
spark = SparkSession.builder.appName("Demo").master("local[*]").getOrCreate()

# Linear Regression: Improving the Model with Categorical Variables

In [ ]:
model_output = "/home/jovyan/work/outputs/models/lr-model/"


## Load Dataset
Let's load the clean Airbnb dataset in again 
We created it in the previous notebook, it should exists in `/home/jovyan/work/outputs/airbnb/clean_data`

In [ ]:
file_path = f"/home/jovyan/work/outputs/airbnb/clean_data"
airbnb_df = spark.read.parquet(file_path)
train_df, test_df = airbnb_df.randomSplit([.8, .2], seed=42)

### One Hot Encoder
* Extract all the string variables
* Create a column with Index for each of them to serve as the output of the StringIndexer
* Create a column with OHE for each of them to serve as the output of the One-Hot Encoder
* You need to use [StringIndexer](https://spark.apache.org/docs/latest/api/python/reference/api/pyspark.ml.feature.StringIndexer.html?highlight=stringindexer#pyspark.ml.feature.StringIndexer) in order to map a string column of labels to an ML column of label indices.
*  Then, apply the [OneHotEncoder](https://spark.apache.org/docs/latest/api/python/reference/api/pyspark.ml.feature.OneHotEncoder.html?highlight=onehotencoder#pyspark.ml.feature.OneHotEncoder) to the output of the StringIndexer.

In [ ]:
from pyspark.ml.feature import StringIndexer

df = spark.createDataFrame(
    [(0, "a"), (1, "b"), (2, "c"), (3, "a"), (4, "a"), (5, "c")],
    ["id", "category"])

print("original")
df.show()
indexer = StringIndexer(inputCol="category", outputCol="categoryIndex")
indexed = indexer.fit(df).transform(df)
print("indexed")
indexed.show()

In [ ]:
from pyspark.ml.feature import OneHotEncoder

df = spark.createDataFrame([
    (0.0, 1.0),
    (1.0, 0.0),
    (2.0, 1.0),
    (0.0, 2.0),
    (0.0, 1.0),
    (2.0, 0.0)
], ["categoryIndex1", "categoryIndex2"])

print("original")
df.show()

encoder = OneHotEncoder(inputCols=["categoryIndex1", "categoryIndex2"],
                        outputCols=["categoryVec1", "categoryVec2"])
model = encoder.fit(df)
encoded = model.transform(df)
encoded.show()

In [ ]:
from pyspark.ml.feature import OneHotEncoder, StringIndexer

categorical_cols = [field for (field, dataType) in train_df.dtypes if dataType == "string"]
index_output_cols = [x + "Index" for x in categorical_cols]
ohe_output_cols = [x + "OHE" for x in categorical_cols]

string_indexer = StringIndexer(inputCols=<TODO>, outputCols=<TODO>, handleInvalid="skip")
ohe_encoder = OneHotEncoder(inputCols=<TODO>, outputCols=<TODO>)

## Vector Assembler
Now you should combine our OHE categorical features with our numeric features.
 * Extract numeric columns (except price, since it is the target one)
 * Use Vector Assembler once again to create the features vector

In [ ]:
from pyspark.ml.feature import VectorAssembler

numeric_cols = [field for (field, dataType) in train_df.dtypes if ((dataType == "double") & (field != "price"))]
assembler_inputs = ohe_output_cols + numeric_cols
vec_assembler = VectorAssembler(inputCols=<TODO>, outputCol=<TODO>)

## Linear Regression
* Build Linear Regression object with price as label

In [ ]:
from pyspark.ml.regression import LinearRegression

lr = LinearRegression(labelCol=<TODO>, featuresCol=<TODO>)

## Pipeline
 A [Pipeline](https://spark.apache.org/docs/latest/api/python/reference/api/pyspark.ml.Pipeline.html?highlight=pipeline#pyspark.ml.Pipeline) is a way of organizing all of the steps.

In [ ]:
from pyspark.ml import Pipeline

#stages should be indexer, ohe, vectorizing and finally linear regression
stages = [<TODO>]
pipeline = Pipeline(stages=<TODO>)

pipeline_model = pipeline.fit(<TODO>)

## Saving Models
Training a model may be costly, so we can save it in case our cluster goes down so we don't have to recompute our results.

In [ ]:
pipeline_model.write().overwrite().save(model_output)

In [ ]:
model_output

## Loading models
If all your transformers/estimators are set into a Pipeline, and stored like that, you can always load the generic `PipelineModel` back in. Otherwise, you may need to know which kind of model was stored (LinearRegression, LogisticRegression etc...)

In [ ]:
from pyspark.ml import PipelineModel

saved_pipeline_model = PipelineModel.load(<TODO>)

## Apply the Model to Test Set

In [ ]:
pred_df = saved_pipeline_model.transform(<TODO>)

display(pred_df.select("features", "price", "prediction"))

## Evaluate the Model

In [ ]:
display(pred_df.select("price", "prediction"))

In [ ]:
from pyspark.ml.evaluation import RegressionEvaluator

regression_evaluator = RegressionEvaluator(predictionCol=<TODO>, labelCol=<TODO>, metricName=<TODO>)

rmse = regression_evaluator.evaluate(pred_df)
r2 = regression_evaluator.setMetricName(<TODO>).evaluate(pred_df)
print(f"RMSE is {rmse}")
print(f"R2 is {r2}")